# 02 — Preprocessing

**Input:** `data/raw/raw_data.csv`  
**Output:** `data/processed/processed_data.csv`

Steps:
1. Drop `ID`
2. Impute missing values
3. Outlier detection & removal (IQR method)
4. Encode categoricals
5. Feature scaling (StandardScaler)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/raw_data.csv')
print('Raw shape:', df.shape)
df.head()

## Step 1 — Drop Unused Columns

In [ ]:
df = df.drop(columns=['ID'], errors='ignore')
print('After drop:', df.shape)

## Step 2 — Impute Missing Values

In [ ]:
print('Missing before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Numeric: fill with median
num_cols = df.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical: fill with mode
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('\nMissing after imputation:', df.isnull().sum().sum())

## Step 3 — Outlier Detection & Removal (IQR)

Visualise boxplots before removal, then apply IQR-based capping.

In [ ]:
# --- Boxplots BEFORE handling ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Age', 'Work_Experience', 'Family_Size']):
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(f'{col} — Before')
plt.suptitle('Boxplots Before Outlier Handling')
plt.tight_layout()
plt.show()

In [ ]:
# IQR Capping (Winsorization) — cap outliers at 1.5*IQR fences
outlier_cols = ['Age', 'Work_Experience', 'Family_Size']

for col in outlier_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = df[col].shape[0]
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    # Cap (winsorise) instead of drop — preserves row count
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'{col}: {outliers} outliers capped  (lower={lower:.1f}, upper={upper:.1f})')

In [ ]:
# --- Boxplots AFTER handling ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Age', 'Work_Experience', 'Family_Size']):
    sns.boxplot(y=df[col], ax=ax, color='seagreen')
    ax.set_title(f'{col} — After')
plt.suptitle('Boxplots After Outlier Capping')
plt.tight_layout()
plt.show()

## Step 4 — Encode Categoricals

In [ ]:
le = LabelEncoder()

# Ordinal encoding for Spending_Score
spending_map = {'Low': 0, 'Average': 1, 'High': 2}
df['Spending_Score'] = df['Spending_Score'].map(spending_map)

# Binary encoding
binary_map = {'Male': 1, 'Female': 0, 'Yes': 1, 'No': 0}
for col in ['Gender', 'Ever_Married', 'Graduated']:
    df[col] = df[col].map(binary_map)

# Label encode remaining objects (Profession, Var_1)
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

print('Dtypes after encoding:')
print(df.dtypes)
df.head()

## Step 5 — Feature Scaling (StandardScaler)

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

print('Scaled stats (mean ≈ 0, std ≈ 1):')
df_scaled.describe().round(2)

## Correlation Heatmap (after processing)

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df_scaled.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap (Processed Data)')
plt.tight_layout()
plt.show()

## Save Processed Data

In [ ]:
df_scaled.to_csv('../data/processed/processed_data.csv', index=False)
print(f'Saved processed_data.csv — shape: {df_scaled.shape}')